## **1. Setting**

In [1]:
# 1. RunPod 패키지 설치 (NumPy 충돌 해결)
%pip install \
    "transformers==4.44.2" \
    "datasets==2.19.0" \
    "trl==0.11.4" \
    "tokenizers==0.19.1" \
    "peft==0.12.0" \
    "huggingface-hub==0.28.1" \
    "accelerate==0.34.2" \
    "numpy==1.26.4" \
    "scikit-learn" \
    "tqdm" \
    "rich" \
    "hf_transfer"

%pip install --force-reinstall "numpy==1.26.4" "pyarrow==16.1.0"

Note: you may need to restart the kernel to use updated packages.
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 203.4 MB/s  0:00:00m0:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
  Attempting uninstall: pyarrow━━━━━━━━━━━━━━━━━ 0/2 [numpy]
    Found existing installation: pyarrow 26.0.1m 0/2 [numpy]
    Uninstalling pyarrow-26.0.1:━━━━━━━━━━━━ 0/2 [numpy]
      Successfully uninstalled pyarrow-26.0.1 0/2 [numpy]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pyarrow]m1/2 [pyarrow]
Note: you may need to restart the kernel to use updated packages.


In [2]:
# 2. 라이브러리 불러오기
import gc
import os
import re
from typing import Dict, List

import torch
from datasets import Dataset, load_dataset
from huggingface_hub import HfApi, login
from peft import AutoPeftModelForCausalLM, LoraConfig, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from trl import SFTConfig, SFTTrainer
import torch

# 5. Hugging Face 로그인
# Hugging Face 로그인 (새 토큰은 HF_TOKEN 환경변수 또는 실행 중 보안 입력창에서 받음)
# 주의: Hugging Face 토큰을 노트북 코드에 직접 붙여넣거나 출력하지 마세요
from getpass import getpass
from huggingface_hub import login, HfApi
HF_TOKEN = (os.getenv("HF_TOKEN") or "").strip()
if not HF_TOKEN:
    HF_TOKEN = getpass("새 Hugging Face access token 입력 (입력값 숨김): ").strip()
if not HF_TOKEN:
    raise ValueError("HF_TOKEN이 필요합니다. 새 토큰을 입력하거나 환경변수 HF_TOKEN을 설정하세요.")
login(token=HF_TOKEN)
HF_USERNAME = (os.getenv("PILATES_HF_USERNAME") or HfApi().whoami(token=HF_TOKEN)["name"]).strip()
print("Hugging Face 로그인 확인:", HF_USERNAME)  # 토큰은 출력하지 않음

# 3. RunPod GPU 환경 확인
print("CUDA 사용 가능:", torch.cuda.is_available())
print("GPU 개수:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

# 4. 최대 시퀀스 길이 설정
max_seq_length = 8192

# 6. Pilates 학습 데이터 및 모델 경로 설정
DATASET_ID = os.getenv("PILATES_DATASET_REPO", f"{HF_USERNAME}/pilates-function-calling-datasets-korean-v2")
BASE_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
OUTPUT_DIR = "qwen-2.5-7b-function-calling-v2"
ADAPTER_PATH = OUTPUT_DIR
MERGED_MODEL_PATH = f"{OUTPUT_DIR}/merged"
MODEL_NAME = "Qwen2.5-7B-Instruct-pilates-function-calling-v2"
MODEL_REPO_ID = f"{HF_USERNAME}/{MODEL_NAME}"


CUDA 사용 가능: True
GPU 개수: 1
GPU 0: NVIDIA A100-SXM4-80GB


새 Hugging Face access token 입력 (입력값 숨김):  ········


Hugging Face 로그인 확인: somsom317


## **2. Data Preprocessing**

In [3]:
# 1. 데이터셋 불러오기
dataset = load_dataset(DATASET_ID)
assert "train" in dataset and "test" in dataset, "Part1의 V2 train/test 분리가 필요합니다."

# 2. Train/Test 분할 비율 설정
test_ratio = 0.2
total_len = len(dataset["train"]) + len(dataset["test"])
test_size = len(dataset["test"])

# 3. 데이터 인덱스 분리
test_indices = list(range(len(dataset["test"])))
train_indices = list(range(len(dataset["train"])))

# 4. OpenAI 메시지 형식으로 변환
def format_conversations(sample):
    return {
        "messages": [
            {"role": "system", "content": sample["system_prompt"]},
            *sample["messages"]
        ]
    }
    
# 5. Train/Test 데이터 변환
train_dataset = [format_conversations(dataset["train"][i]) for i in train_indices]
test_dataset = [format_conversations(dataset["test"][i]) for i in test_indices]

# 6. Hugging Face Dataset 객체로 변환
train_dataset = Dataset.from_list(train_dataset)
test_dataset = Dataset.from_list(test_dataset)

# 7. 데이터 분할 결과 확인
print(f"\n전체 데이터 분할 결과: Train {len(train_dataset)}개, Test {len(test_dataset)}개")

print(train_dataset[min(5, len(train_dataset) - 1)]["messages"])

Generating train split:   0%|          | 0/64 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/16 [00:00<?, ? examples/s]


전체 데이터 분할 결과: Train 64개, Test 16개
[{'content': '당신은 글로리 필라테스의 AI 상담사입니다. 정확하고 자연스럽게 상담하십시오.\n\n행동 선택 규칙:\n- 이미 알고 있는 정보로 답할 수 있으면 바로 답하고, 실제 센터/회원 DB 정보가 필요하면 Tool을 사용합니다.\n- 선택 인자가 누락되어도 전체 조회가 가능하면 강사·요일 등을 불필요하게 되묻지 않고 바로 Tool을 호출합니다.\n- 모호한 지시 대상이나 서로 충돌하는 조건처럼 Tool·대화 맥락으로도 해소할 수 없는 필수 정보가 있을 때만 한 가지를 되묻습니다.\n- 사용자가 그 모호함을 풀어주면 다시 물어보지 말고 곧바로 필요한 작업을 처리합니다.\n- 월요일·수요일처럼 양립 가능한 복수 조건은 필요 시 여러 Tool을 호출합니다.\n- 이전 Tool 결과만으로 답할 수 있으면 불필요하게 재조회하지 않습니다.\n- 예약 생성·취소·결제 등 Tool에 없는 작업은 실행했다고 말하지 않습니다.\n- 함수 호출 전 별도의 자연어 안내 멘트를 만들지 않습니다.\n\n로그인한 사용자의 현재 ID: U001\n오늘 날짜: 2026-11-02\n\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>\n{"type": "function", "function": {"name": "search_policy_info", "description": "센터의 예약 오픈, 취소, 대기, 출석, 환불 및 운영시간에 관한 정책과 약관을 조회합니다. \'대기 예약 가능해?\', \'취소하면 횟수 돌려받아?\'처럼 규정을 묻는 질문에 사용합니다. 실제 빈자리나 회원 본인의 예약 내역을 조회하는 함수는 아닙니다.", "parameters": {"type": 

## **3. Fine Tuning**

In [8]:
# 1. Qwen 모델 ID 설정
model_id = BASE_MODEL_ID

# 2. 모델 및 Tokenizer 불러오기
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

# 3. Chat Template 적용 결과 확인
text = tokenizer.apply_chat_template(
    train_dataset[14]["messages"],
    tokenize=False,
    add_generation_prompt=False
)
print(text)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

<|im_start|>system
당신은 글로리 필라테스의 AI 상담사입니다. 정확하고 자연스럽게 상담하십시오.

행동 선택 규칙:
- 이미 알고 있는 정보로 답할 수 있으면 바로 답하고, 실제 센터/회원 DB 정보가 필요하면 Tool을 사용합니다.
- 선택 인자가 누락되어도 전체 조회가 가능하면 강사·요일 등을 불필요하게 되묻지 않고 바로 Tool을 호출합니다.
- 모호한 지시 대상이나 서로 충돌하는 조건처럼 Tool·대화 맥락으로도 해소할 수 없는 필수 정보가 있을 때만 한 가지를 되묻습니다.
- 사용자가 그 모호함을 풀어주면 다시 물어보지 말고 곧바로 필요한 작업을 처리합니다.
- 월요일·수요일처럼 양립 가능한 복수 조건은 필요 시 여러 Tool을 호출합니다.
- 이전 Tool 결과만으로 답할 수 있으면 불필요하게 재조회하지 않습니다.
- 예약 생성·취소·결제 등 Tool에 없는 작업은 실행했다고 말하지 않습니다.
- 함수 호출 전 별도의 자연어 안내 멘트를 만들지 않습니다.

로그인한 사용자의 현재 ID: U009
오늘 날짜: 2026-11-02

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "get_available_classes", "description": "예약이 오픈된 주차에서 실제 잔여석이 1석 이상인 수업만 조회합니다. '빈자리', '자리 남았어?', '예약 가능한 시간'일 때 사용합니다. 수업 예약 현황을 집계하므로 별도의 내 예약 조회 없이 일반 잔여석을 확인할 수 있지만, 회원의 이용권 사용 자격이나 같은 날짜 중복 예약 여부는 판단하지 않습니다. '내 이용권으로 김민지 선생님 수업 예약 가능?'이라면 get_my_member

In [9]:
# 4. LoRA 및 SFTConfig 설정
peft_config = LoraConfig(
        lora_alpha=32,
        lora_dropout=0.1,
        r=16,
        bias="none",
        target_modules=["q_proj", "v_proj", "gate_proj", "up_proj", "down_proj"],
        task_type="CAUSAL_LM",
)

args = SFTConfig(
    output_dir=OUTPUT_DIR,                     # 저장될 디렉토리와 저장소 ID
    num_train_epochs=3,                      # 학습할 총 에포크 수
    per_device_train_batch_size=2,           # GPU당 배치 크기
    gradient_accumulation_steps=2,           # 그래디언트 누적 스텝 수
    gradient_checkpointing=True,             # 메모리 절약을 위한 체크포인팅
    optim="adamw_torch_fused",               # 최적화기
    logging_steps=10,                        # 로그 기록 주기
    save_strategy="steps",                   # 저장 전략
    save_steps=50,                           # 저장 주기
    bf16=True,                               # bfloat16 사용
    learning_rate=1e-4,                      # 학습률
    max_grad_norm=0.3,                       # 그래디언트 클리핑
    warmup_ratio=0.03,                       # 워밍업 비율
    lr_scheduler_type="constant",            # 고정 학습률
    push_to_hub=False,                       # 허브 업로드 안 함
    remove_unused_columns=False,
    dataset_kwargs={"skip_prepare_dataset": True},
    report_to=None
)

In [10]:
# 5. Assistant 응답 전용 전처리 함수 구성 

def collate_fn(batch):
    new_batch = {
        "input_ids": [],
        "attention_mask": [],
        "labels": []
    }

    # Qwen ChatML 시작 및 종료 토큰 정의함
    start_token = "<|im_start|>"
    end_token = "<|im_end|>"

    # Assistant 시작 위치와 종료 위치 탐색용 토큰 정의함
    assistant_prefix = f"{start_token}assistant\n"
    assistant_tokens = tokenizer.encode(assistant_prefix, add_special_tokens=False)
    end_tokens = tokenizer.encode(end_token, add_special_tokens=False)

    for example in batch:
        messages = example["messages"]

        # Qwen Chat Template 통일 (assistant/user/tool 역할 처리)
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )

        # ChatML 문자열을 토큰화함
        tokenized = tokenizer(
            prompt,
            truncation=True,
            max_length=max_seq_length,
            add_special_tokens=False,
            padding=False,
            return_tensors=None
        )
        input_ids = tokenized["input_ids"]
        attention_mask = tokenized["attention_mask"]
        labels = [-100] * len(input_ids)

        # Assistant 응답과 종료 토큰에만 레이블을 부여함
        i = 0
        n = len(input_ids)
        while i <= n - len(assistant_tokens):
            if input_ids[i:i + len(assistant_tokens)] == assistant_tokens:
                start_idx = i + len(assistant_tokens)
                end_idx = start_idx
                # Assistant 응답의 종료 토큰 위치를 탐색함
                while end_idx <= n - len(end_tokens):
                    if input_ids[end_idx:end_idx + len(end_tokens)] == end_tokens:
                        end_idx += len(end_tokens)
                        break
                    end_idx += 1
                # 응답 본문과 종료 토큰을 학습 레이블로 지정함
                for j in range(start_idx, end_idx):
                    labels[j] = input_ids[j]
                i = end_idx
            else:
                i += 1

        new_batch["input_ids"].append(input_ids)
        new_batch["attention_mask"].append(attention_mask)
        new_batch["labels"].append(labels)

    # 배치별 길이를 맞추고 Tensor로 변환함
    max_len = max(len(ids) for ids in new_batch["input_ids"])
    for idx in range(len(new_batch["input_ids"])):
        pad_len = max_len - len(new_batch["input_ids"][idx])
        new_batch["input_ids"][idx].extend([tokenizer.pad_token_id] * pad_len)
        new_batch["attention_mask"][idx].extend([0] * pad_len)
        new_batch["labels"][idx].extend([-100] * pad_len)

    for k in new_batch:
        new_batch[k] = torch.tensor(new_batch[k])

    return new_batch
# 6턴 대화가 max_seq_length를 지나치게 초과하지 않는지 검증
lengths = [len(tokenizer.apply_chat_template(x['messages'], tokenize=True,
                                          add_generation_prompt=False)) for x in train_dataset]
too_long = sum(n > max_seq_length for n in lengths)
print(f'학습 대화 길이: 최대 {max(lengths)} / 평균 {sum(lengths)/len(lengths):.0f} 토큰')
print(f'8192 토큰 초과 대화: {too_long}/{len(lengths)}')
if too_long > len(lengths) * 0.10:
    raise ValueError('학습 대화 10% 초과가 max_seq_length를 넘음. Part1 MAX_TOOL_ITEMS를 줄여 재생성하세요.')


학습 대화 길이: 최대 9211 / 평균 6693 토큰
8192 토큰 초과 대화: 3/64


In [11]:
# 6. Fine-tuning 학습 및 저장
trainer = SFTTrainer(
    model=model,
    args=args,
    max_seq_length=max_seq_length,  # 최대 시퀀스 길이 설정
    train_dataset=train_dataset,
    data_collator=collate_fn,
    peft_config=peft_config,
)

# 학습을 시작함
trainer.train()   # 학습 실행함 (허브 업로드 여부는 push_to_hub 설정을 따름)

# 학습 완료 모델을 저장함
trainer.save_model()   # 최종 모델을 저장함

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:283: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences betwee

Step,Training Loss
10,0.627700
20,0.360400
30,0.269000
40,0.199600


## **4. Testing**

In [12]:
# 1. 테스트 대화에서 Assistant 응답별 Input/Label 추출
# 학습 collate_fn과 동일한 Qwen Chat Template 사용 (기존 임의 줄바꿈 제거)
def to_chatml(messages):
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )

def extract_examples(data):
    messages = data["messages"] if isinstance(data,dict) else data
    examples=[]
    for idx,msg in enumerate(messages):
        if msg["role"]!="assistant":continue
        previous_messages=messages[:idx]
        input_text=tokenizer.apply_chat_template(
            previous_messages,tokenize=False,add_generation_prompt=True
        )
        examples.append({"input":input_text,"label":msg["content"].strip()})
    return examples
prompt_lst=[]; label_lst=[]
for item in test_dataset:
    for ex in extract_examples(item):
        prompt_lst.append(ex["input"]);label_lst.append(ex["label"])
print(f"원본 테스트 대화 수: {len(test_dataset)}")
print(f"Assistant 응답별 테스트 샘플 수: {len(prompt_lst)}")
if prompt_lst:
    print("\n[Input 예시]",prompt_lst[0][-1300:])
    print("\n[Label 예시]",label_lst[0])


원본 테스트 대화 수: 16
Assistant 응답별 테스트 샘플 수: 130

[Input 예시] ": "string", "description": "HH:MM 형식입니다. between에서는 수업 시작 시각 검색 범위의 상한(양 끝 포함), 그 외에는 수업 자체 종료 시각의 상한(이하)입니다."}, "time_filter": {"type": "string", "enum": ["exact", "after", "at_or_after", "before", "at_or_before", "between"], "description": "수업의 시작 시각 기준: exact=정각 시작(==), before='이전' 미만(<), at_or_before='까지' 이하(<=), after='초과' 기준 시각 제외(>), at_or_after='부터' 이상(>=), between='A부터 B까지' 시작 시각 구간 양끝 포함입니다. 예: 3시 이전은 15:00 제외, 3시부터는 15:00 포함. 생략하면 exact입니다."}}, "required": [], "additionalProperties": false}}}
{"type": "function", "function": {"name": "get_my_memberships", "description": "로그인한 회원 본인의 보유 이용권, 상품 유형, 유효기간, 남은 횟수를 조회합니다. 단일 및 콤보 이용권을 모두 반환하며 콤보 상품은 remaining_sessions_by_type에서 '7:1 레슨'(그룹), '2:1 듀엣 레슨'(듀엣), '1:1 개인 레슨'(개인)의 남은 횟수를 각각 확인합니다. 상품 구매 가능 목록은 get_membership_info, 실제 빈자리는 get_available_classes, 이미 예약한 날짜는 get_my_bookings로 확인합니다. 로그인 회원은 내부에서 결정하며 user_id 인자는 받지 않습니다.", "parameters": {"type": "object", "properties

`
<|im_start|>system
너는 필라테스 챗봇이야.<|im_end|>

<|im_start|>user
이용권 확인해줘.<|im_end|>

<|im_start|>assistant
10회 남았습니다.<|im_end|>

<|im_start|>user
그럼 내일 예약해줘.<|im_end|>
<|im_start|>assistant

`

In [15]:

# 이전에 로딩된 모델이 있다면 메모리 정리
for name in ["ft_pipe", "fine_tuned_model", "base_pipe", "base_model"]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

# GPU 메모리 확인
free_mem, total_mem = torch.cuda.mem_get_info()
print(f"GPU 여유 메모리: {free_mem / 1024**3:.2f} GB")
print(f"GPU 전체 메모리: {total_mem / 1024**3:.2f} GB")


# 실제 상담 답변이 있는 샘플 5개 선택
def is_valid_answer(label):
    label = label.strip()
    exclude = [
        "<tool_call>",
        "상담을 종료하겠습니다",
        "지금까지 글로리 필라테스",
        "확인하고 안내드리겠습니다",
        "조회해 드리겠습니다",
        "잠시만 기다려 주세요"
    ]
    return len(label) >= 50 and not any(x in label for x in exclude)


# 함수 호출 2개, 비호출 3개
_tool = [
    i for i, x in enumerate(label_lst)
    if "<tool_call>" in x
]

_no_tool = [
    i for i, x in enumerate(label_lst)
    if "<tool_call>" not in x
]

selected_indices = (_tool[:2] + _no_tool[:3])[:5]

test_prompts = [prompt_lst[i] for i in selected_indices]
test_labels = [label_lst[i] for i in selected_indices]

eos_token = tokenizer(
    "<|im_end|>",
    add_special_tokens=False
)["input_ids"][0]


def test_inference(pipe, prompt):
    outputs = pipe(
        prompt,
        max_new_tokens=1024,
        eos_token_id=eos_token,
        do_sample=False,
        return_full_text=False
    )
    return outputs[0]["generated_text"].split("<|im_end|>")[0].strip()


# 1. Fine-Tuned Model 평가
print("\nFine-Tuned Model 로딩 중...")

fine_tuned_model = AutoPeftModelForCausalLM.from_pretrained(
    ADAPTER_PATH,
    device_map={"": 0},
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

fine_tuned_model.eval()

ft_pipe = pipeline(
    "text-generation",
    model=fine_tuned_model,
    tokenizer=tokenizer
)

with torch.inference_mode():
    ft_responses = [
        test_inference(ft_pipe, p)
        for p in test_prompts
    ]

# GPU 메모리 정리
del ft_pipe, fine_tuned_model
gc.collect()
torch.cuda.empty_cache()

print("Fine-Tuned 평가 완료 / GPU 메모리 정리 완료")


# 2. Base Model 평가
print("\nBase Model 로딩 중...")

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    device_map={"": 0},
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

base_model.eval()

base_pipe = pipeline(
    "text-generation",
    model=base_model,
    tokenizer=tokenizer
)

with torch.inference_mode():
    base_responses = [
        test_inference(base_pipe, p)
        for p in test_prompts
    ]

# GPU 메모리 정리
del base_pipe, base_model
gc.collect()
torch.cuda.empty_cache()

print("Base Model 평가 완료")


# 3. 결과 비교
for i, (prompt, label) in enumerate(
    zip(test_prompts, test_labels)
):
    print(f"\n[{i+1}] INPUT:\n{prompt[-1500:]}")

    print(f"\nFINE-TUNED RESPONSE:\n{ft_responses[i]}")

    print(f"\nBASE RESPONSE:\n{base_responses[i]}")

    print(f"\nLABEL:\n{label}")
    print()


GPU 여유 메모리: 61.47 GB
GPU 전체 메모리: 79.25 GB

Fine-Tuned Model 로딩 중...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The model 'PeftModelForCausalLM' is not supported for text-generation. Supported models are ['BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'ElectraForCausalLM', 'ErnieForCausalLM', 'FalconForCausalLM', 'FuyuForCausalLM', 'GemmaForCausalLM', 'Gemma2ForCausalLM', 'GitForCausalLM', 'GPT2LMHeadModel', 'GPT2LMHeadModel', 'GPTBigCodeForCausalLM', 'GPTNeoForCausalLM', 'GPTNeoXForCausalLM', 'GPTNeoXJapaneseForCausalLM', 'GPTJForCausalLM', 'JambaForCausalLM', 'JetMoeForCausalLM', 'LlamaForCausalLM', 'MambaForCausalLM', 'Mamba2ForCausalLM', 'MarianForCausalLM', 'MBartForCausalLM', 'MegaForCausalLM', 'MegatronBertForCausalLM', 'MistralForCausalLM', 'MixtralForCausal

Fine-Tuned 평가 완료 / GPU 메모리 정리 완료

Base Model 로딩 중...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Base Model 평가 완료

[1] INPUT:
='15:00'). time_filter와 함께 지정합니다."}, "end_time": {"type": "string", "description": "HH:MM 형식입니다. between에서는 수업 시작 시각 검색 범위의 상한(양 끝 포함), 그 외에는 수업 자체 종료 시각의 상한(이하)입니다."}, "time_filter": {"type": "string", "enum": ["exact", "after", "at_or_after", "before", "at_or_before", "between"], "description": "수업의 시작 시각 기준: exact=정각 시작(==), before='이전' 미만(<), at_or_before='까지' 이하(<=), after='초과' 기준 시각 제외(>), at_or_after='부터' 이상(>=), between='A부터 B까지' 시작 시각 구간 양끝 포함입니다. 예: 3시 이전은 15:00 제외, 3시부터는 15:00 포함. 생략하면 exact입니다."}}, "required": [], "additionalProperties": false}}}
{"type": "function", "function": {"name": "get_my_memberships", "description": "로그인한 회원 본인의 보유 이용권, 상품 유형, 유효기간, 남은 횟수를 조회합니다. 단일 및 콤보 이용권을 모두 반환하며 콤보 상품은 remaining_sessions_by_type에서 '7:1 레슨'(그룹), '2:1 듀엣 레슨'(듀엣), '1:1 개인 레슨'(개인)의 남은 횟수를 각각 확인합니다. 상품 구매 가능 목록은 get_membership_info, 실제 빈자리는 get_available_classes, 이미 예약한 날짜는 get_my_bookings로 확인합니다. 로그인 회원은 내부에서 결정하며 user_id 인자는 받지 않습니다.", "parameters": {"

In [16]:
# 4. 모델 병합 및 업로드 경로 설정
base_model_path = BASE_MODEL_ID
adapter_path = ADAPTER_PATH
merged_model_path = MERGED_MODEL_PATH

# 디바이스 설정함
device_arg = {"device_map": "auto"}

# 베이스 모델 불러옴
print(f"Loading base model from: {base_model_path}")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    return_dict=True,
    torch_dtype=torch.float16,
    **device_arg
)

# LoRA 어댑터를 불러와 베이스 모델에 병합함
print(f"Loading and merging PEFT from: {adapter_path}")
model = PeftModel.from_pretrained(base_model, adapter_path, **device_arg)
model = model.merge_and_unload()

# Tokenizer 불러옴
tokenizer = AutoTokenizer.from_pretrained(base_model_path)

# 병합한 모델 및 Tokenizer를 저장함
print(f"Saving merged model to: {merged_model_path}")
model.save_pretrained(merged_model_path)
tokenizer.save_pretrained(merged_model_path)
print("✅ 모델과 토크나이저 저장 완료")

# 5. Hugging Face Hub 업로드
api = HfApi()

username = HF_USERNAME


api.create_repo(
    token=HF_TOKEN,
    repo_id=MODEL_REPO_ID,
    repo_type="model",
    exist_ok=True
)

api.upload_folder(
    token=HF_TOKEN,
    repo_id=MODEL_REPO_ID,
    folder_path=MERGED_MODEL_PATH,
)

Loading base model from: Qwen/Qwen2.5-7B-Instruct


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading and merging PEFT from: qwen-2.5-7b-function-calling-v2
Saving merged model to: qwen-2.5-7b-function-calling-v2/merged
✅ 모델과 토크나이저 저장 완료


  0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.88G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.33G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/somsom317/Qwen2.5-7B-Instruct-pilates-function-calling-v2/commit/b85c43cf24f16c3c27b00d7f3f8d89cde4fe094f', commit_message='Upload folder using huggingface_hub', commit_description='', oid='b85c43cf24f16c3c27b00d7f3f8d89cde4fe094f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/somsom317/Qwen2.5-7B-Instruct-pilates-function-calling-v2', endpoint='https://huggingface.co', repo_type='model', repo_id='somsom317/Qwen2.5-7B-Instruct-pilates-function-calling-v2'), pr_revision=None, pr_num=None)